In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from data_preprocess import FeatureSelection

In [10]:
class NaiveBayesGaussian:
    def __init__(self):
        self.stats = {}
        self.priors = {}
        self.classes = None

    def fit(self, X, y):
        self.classes = np.unique(y)
        for c in self.classes:
            X_c = X[y == c]
            self.priors[c] = X_c.shape[0] / X.shape[0]
            self.stats[c] = {
                'mean': np.mean(X_c, axis=0),
                'var': np.var(X_c, axis=0) + 1e-4 
            }

    def predict(self, X):
        preds = [self._predict_one(x) for x in X]
        return np.array(preds)

    def _predict_one(self, x):
        posteriors = []
        for c in self.classes:
            prior = np.log(self.priors[c])
            mean = self.stats[c]['mean']
            var = self.stats[c]['var']
            likelihood = -0.5 * np.sum(np.log(2 * np.pi * var))
            likelihood -= 0.5 * np.sum(((x - mean)**2) / var)
            posteriors.append(prior + likelihood)
        return self.classes[np.argmax(posteriors)]

In [ ]:
class PCA:
    def __init__(self, data_values: np.ndarray):
        self.data_values = data_values
    
    def shift_values(self) -> np.ndarray:
        means = np.mean(self.data_values, axis=0)
        return self.data_values - means
    
    def compute_fprime(self, k=50):
        shifted_values = self.shift_values()
        cov_matrix = np.cov(shifted_values, rowvar=False)
        
        eigenvals, eigenvectors = np.linalg.eigh(cov_matrix)
        
        sorted_idx = np.argsort(eigenvals)[::-1]
        eigenvectors = eigenvectors[:, sorted_idx]
        
        Q = eigenvectors[:, :k]
        return shifted_values @ Q

In [ ]:
X, y = fetch_openml('mnist_784', version=1, as_frame=False, return_X_y=True)
X = X / 255.0 

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
nb_all = NaiveBayesGaussian()
nb_all.fit(X_train, y_train)
print(f"Accuracy: {accuracy_score(y_test, nb_all.predict(X_test)):.4f}")

Accuracy: 0.7187


In [12]:
fs = FeatureSelection()
X_fs = fs.select_top_variance(X, k=100)
X_train, X_test, y_train, y_test = train_test_split(X_fs, y, test_size=0.2)
nb_fs = NaiveBayesGaussian()
nb_fs.fit(X_train, y_train)
print(f"Accuracy: {accuracy_score(y_test, nb_fs.predict(X_test)):.4f}")

Accuracy: 0.7352


In [13]:
pca_tool = PCA(X)
X_pca = pca_tool.compute_fprime(k=50)
X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2)
nb_pca = NaiveBayesGaussian()
nb_pca.fit(X_train, y_train)
print(f"Accuracy: {accuracy_score(y_test, nb_pca.predict(X_test)):.4f}")

Accuracy: 0.8755
